# Deep Learning — Week 2: Logistic Regression & Binary Classification

**Student Version**

---

## 이번 주 학습 목표
1. 이진분류 문제가 회귀와 어떻게 다른지 설명할 수 있다
2. 시그모이드 함수의 역할과 BCE Loss의 의미를 이해한다
3. NumPy로 로지스틱 회귀를 직접 구현하고 결정경계를 시각화할 수 있다
4. Softmax를 통해 다중 클래스로 확장되는 방향을 이해한다

## 사전 학습 안내
> 영상(약 40분)을 먼저 보고 오세요.  
> **Part 1 전체를 직접 실행**해보는 것이 사전 학습 과제입니다.

## 지난 주와의 연결
> 1주차에서 배운 흐름: **Forward (출력예측) → Loss (오차계산) → Gradient (편미분 경사도) → Update (파라메터 + or -)**  
> 이번 주도 같은 흐름입니다. 달라지는 것은 Loss 함수와 출력 함수뿐입니다.

---
# Part 1. 강의 내용 (영상과 동일)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

np.random.seed(42)

: 

## 1. 이진분류란?

### 회귀(Regression) vs 분류(Classification)

1주차에서 배운 **선형회귀**는 연속적인 숫자를 예측하는 문제였습니다.
- 집 크기 → **가격** (얼마짜리인가?)
- 공부 시간 → **점수** (몇 점인가?)

**이진분류**는 두 가지 중 하나를 예측하는 문제입니다.
- 이메일 → **스팸 / 정상** (1 or 0)
- 종양 → **암이다 / 아니다** (1 or 0)
- 신용카드 거래 → **사기 / 정상** (1 or 0)

### 왜 선형회귀를 그대로 쓰면 안 되나?

분류 문제에서 출력은 **확률** — 즉 0과 1 사이의 값이어야 합니다.  
선형회귀의 출력 $wx + b$ 는 어떤 값이든 나올 수 있습니다 (-100만원, 5000만원).  
이것을 **0~1 사이로 눌러주는 함수**가 필요합니다 → **시그모이드(Sigmoid)**

In [ ]:
# 선형회귀로 분류를 시도하면 어떤 문제가 생기는지 시각화
np.random.seed(42)
x_cls = np.concatenate([np.random.randn(20) - 2,   # class 0: 왼쪽
                         np.random.randn(20) + 2])  # class 1: 오른쪽
y_cls = np.array([0]*20 + [1]*20)

# 선형회귀로 억지로 맞추기
w_lin = np.polyfit(x_cls, y_cls, 1)
x_line = np.linspace(-6, 6, 100)
y_line = np.polyval(w_lin, x_line)

plt.figure(figsize=(9, 4))
plt.subplot(1, 2, 1)
plt.scatter(x_cls, y_cls, c=y_cls, cmap='bwr', edgecolors='k', s=60)
plt.plot(x_line, y_line, 'k--', linewidth=2, label='Linear fit')
plt.axhline(0.5, color='gray', linestyle=':', label='threshold 0.5')
plt.ylim(-0.3, 1.3)
plt.xlabel('x'); plt.ylabel('y')
plt.title('Linear Regression for Classification\n(output can exceed 0~1!)')
plt.legend(fontsize=8)
plt.grid(True, alpha=0.3)

# 시그모이드를 쓰면
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

plt.subplot(1, 2, 2)
plt.scatter(x_cls, y_cls, c=y_cls, cmap='bwr', edgecolors='k', s=60)
plt.plot(x_line, sigmoid(x_line * 2), 'steelblue', linewidth=2, label='Sigmoid fit')
plt.axhline(0.5, color='gray', linestyle=':', label='threshold 0.5')
plt.ylim(-0.3, 1.3)
plt.xlabel('x'); plt.ylabel('P(y=1|x)')
plt.title('Logistic Regression\n(output always in 0~1)')
plt.legend(fontsize=8)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 2. 시그모이드 함수 (Sigmoid Function)

### 정의

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

### 핵심 성질

| 입력 $z$ | 출력 $\sigma(z)$ | 의미 |
|---|--------|---|
| 매우 큰 양수 |&nbsp; ≈ 1 | class 1일 확률 거의 100% |
| 0 | &nbsp; 0.5 | 반반 |
| 매우 큰 음수 | &nbsp; ≈ 0 | class 1일 확률 거의 0% |

### 로지스틱 회귀 모델 전체 흐름

$$z = wx + b \quad \xrightarrow{\text{sigmoid}} \quad \hat{y} = \sigma(z) = P(y=1 \mid x)$$

- $\hat{y} \geq 0.5$ → class 1 예측
- $\hat{y} < 0.5$ → class 0 예측

### 미분 (나중에 역전파에서 필요)

$$\frac{d\sigma}{dz} = \sigma(z)(1 - \sigma(z))$$

자기 자신으로 표현되는 아름다운 형태입니다.

In [ ]:
def sigmoid(z):
    """Sigmoid function: maps any real number to (0, 1)."""
    return 1 / (1 + np.exp(-z))

def sigmoid_grad(z):
    """Derivative of sigmoid."""
    s = sigmoid(z)
    return s * (1 - s)

z = np.linspace(-8, 8, 200)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: sigmoid
axes[0].plot(z, sigmoid(z), color='steelblue', linewidth=2.5)
axes[0].axhline(0.5, color='gray', linestyle='--', alpha=0.7, label='y = 0.5')
axes[0].axhline(0,   color='gray', linestyle=':',  alpha=0.5)
axes[0].axhline(1,   color='gray', linestyle=':',  alpha=0.5)
axes[0].axvline(0,   color='gray', linestyle=':',  alpha=0.5)
axes[0].set_xlabel('z'); axes[0].set_ylabel('sigmoid(z)')
axes[0].set_title('Sigmoid Function  σ(z) = 1 / (1 + e^{-z})')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# 특정 점 표시
for z_pt, color in [(-4, 'tomato'), (0, 'gray'), (4, 'seagreen')]:
    axes[0].annotate(f'σ({z_pt}) = {sigmoid(z_pt):.2f}',
                     xy=(z_pt, sigmoid(z_pt)),
                     xytext=(z_pt + 0.5, sigmoid(z_pt) - 0.12),
                     fontsize=9, color=color,
                     arrowprops=dict(arrowstyle='->', color=color))

# Right: gradient
axes[1].plot(z, sigmoid_grad(z), color='tomato', linewidth=2.5)
axes[1].set_xlabel('z'); axes[1].set_ylabel("sigmoid'(z)")
axes[1].set_title("Sigmoid Derivative  σ'(z) = σ(z)(1 - σ(z))")
axes[1].grid(True, alpha=0.3)
axes[1].annotate('max at z=0\n(= 0.25)', xy=(0, 0.25), xytext=(2, 0.22),
                 fontsize=9, arrowprops=dict(arrowstyle='->'))

plt.tight_layout()
plt.show()

print('sigmoid(-10):', sigmoid(-10))
print('sigmoid(  0):', sigmoid(0))
print('sigmoid( 10):', sigmoid(10))





'''
미분된 sigmoid는 큰 음수와 큰 양수의 구간에서는 기울기가 평평해 미분계수가 0에 가까워짐.
그로 인해 gradient를 구할 때 w = w- lr*dw에서 dw가 0에 가까워져서 w가 거의 업데이트되지 않는 현상(= vanishing gradient)이 발생할 수 있음.
'''

---
## 3. BCE Loss (Binary Cross-Entropy)

### BCE Loss 정의

$$L = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i) \right]$$


### "왜 오차의 제곱(MSE)을 쓰지 않는가?"

분류에서 출력 $\hat{y}$는 0~1 사이의 **확률**입니다.  
정답이 1인데 $\hat{y} = 0.001$ 을 출력했다면, 모델은 **극도로 확신하며 틀린** 것입니다.

이 상황에서 MSE는:  
$(0.001 - 1)^2 \approx 1.0$ — 그냥 "틀렸네" 정도로만 봅니다.

BCE는:  
$-\log(0.001) \approx 6.9$ — **"이 정도로 틀리면 매우 크게 벌점"** 을 줍니다.

> BCE의 핵심: **확신에 찬 값(0 or 1에 가까운 값)을 출력하며 틀릴수록 손실이 폭발적으로 커진다.**  
> $-\log(x)$ 는 $x \to 0$ 일 때 $\infty$ 로 발산하기 때문입니다.

이것이 아래 그래프에서 곡선이 왼쪽으로 갈수록 가파르게 치솟는 이유입니다..

### 직관적으로 이해하기

| 실제값 $y$ | 예측값 $\hat{y}$ | Loss | 해석 |
|---|---|---|---|
| 1 | 0.99 | ≈ 0 | 잘 맞춤 → 패널티 거의 없음 |
| 1 | 0.01 | ≈ 4.6 | 틀림 → 큰 패널티 |
| 0 | 0.01 | ≈ 0 | 잘 맞춤 → 패널티 거의 없음 |
| 0 | 0.99 | ≈ 4.6 | 틀림 → 큰 패널티 |

확신을 갖고 틀릴수록 패널티가 **기하급수적으로** 커집니다.

### Gradient (편미분 결과)

시그모이드 + BCE를 합치면 gradient가 놀랍도록 깔끔해집니다:

$$\frac{\partial L}{\partial w} = \frac{1}{N} X^T (\hat{y} - y), \qquad \frac{\partial L}{\partial b} = \frac{1}{N} \sum (\hat{y}_i - y_i)$$

1주차의 선형회귀 gradient와 **형태가 완전히 같습니다!** (계수 2 제외)  
달라지는 것은 $\hat{y}$ 의 계산 방식뿐입니다.

In [ ]:
# BCE Loss 직관 시각화
y_hat = np.linspace(1e-7, 1 - 1e-7, 300)

bce_y1 = -np.log(y_hat)       # y=1일 때: -log(y_hat)
bce_y0 = -np.log(1 - y_hat)   # y=0일 때: -log(1 - y_hat)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(y_hat, bce_y1, color='steelblue', linewidth=2.5)
axes[0].set_xlabel('Predicted probability (y_hat)')
axes[0].set_ylabel('Loss')
axes[0].set_title('BCE Loss when y = 1\n-log(y_hat)')
axes[0].set_ylim(0, 5)
axes[0].annotate('y_hat=1: loss=0\n(correct!)', xy=(1, 0), xytext=(0.7, 1.5),
                 fontsize=9, arrowprops=dict(arrowstyle='->'))
axes[0].annotate('y_hat→0: loss→∞\n(wrong with confidence!)', xy=(0.05, 3),
                 xytext=(0.2, 3.5), fontsize=9, arrowprops=dict(arrowstyle='->'))
axes[0].grid(True, alpha=0.3)

axes[1].plot(y_hat, bce_y0, color='tomato', linewidth=2.5)
axes[1].set_xlabel('Predicted probability (y_hat)')
axes[1].set_ylabel('Loss')
axes[1].set_title('BCE Loss when y = 0\n-log(1 - y_hat)')
axes[1].set_ylim(0, 5)
axes[1].annotate('y_hat=0: loss=0\n(correct!)', xy=(0, 0), xytext=(0.2, 1.5),
                 fontsize=9, arrowprops=dict(arrowstyle='->'))
axes[1].annotate('y_hat→1: loss→∞\n(wrong with confidence!)', xy=(0.95, 3),
                 xytext=(0.5, 3.5), fontsize=9, arrowprops=dict(arrowstyle='->'))
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 4. 로지스틱 회귀 — NumPy로 구현

### 전체 흐름 (1주차와 비교)

| 단계 | &nbsp;&nbsp;&nbsp; 선형회귀 (1주차)&nbsp;&nbsp;&nbsp; | 로지스틱 회귀 (이번 주) |
|---|---|---|
| Forward | $\hat{y} = Xw + b$ | $\hat{y} = \sigma(Xw + b)$ |
| Loss | MSE | BCE |
| Gradient $\partial L \over \partial w$ | $\frac{2}{N} X^T(\hat{y}-y)$ | $\frac{1}{N} X^T(\hat{y}-y)$ |
| Update | $w \leftarrow w - \alpha {\partial L \over \partial w}$ | 동일 |

코드 구조는 **거의 그대로**입니다. Forward pass에 sigmoid 하나 추가, Loss 함수만 교체.

In [ ]:
# ── 데이터 생성 (2D 이진분류) ─────────────────────────────
# 시험 점수 2개(수학, 영어)로 합격(1) / 불합격(0) 예측
np.random.seed(42)
N = 100

# Class 0: 불합격 (점수 낮은 쪽)
X0 = np.random.randn(N // 2, 2) + np.array([-1.5, -1.5])
# Class 1: 합격 (점수 높은 쪽)
X1 = np.random.randn(N // 2, 2) + np.array([1.5,  1.5])

X = np.vstack([X0, X1]) # stack은 행렬을 위아래로 이어붙여(vertical) 새 행렬을 만듬. class0과 class1을 합쳐서 (100, 2)의 데이터셋을 만듬.
y = np.array([0]*50 + [1]*50).reshape(-1, 1)  # (100, 1)

# 특성 정규화
X_mean, X_std = X.mean(axis=0), X.std(axis=0)
X_norm = (X - X_mean) / X_std

plt.figure(figsize=(6, 5))
plt.scatter(X[y.flatten()==0, 0], X[y.flatten()==0, 1],
            color='tomato', label='Fail (y=0)', edgecolors='k', s=50)
plt.scatter(X[y.flatten()==1, 0], X[y.flatten()==1, 1],
            color='steelblue', label='Pass (y=1)', edgecolors='k', s=50)
plt.xlabel('Math score'); plt.ylabel('English score')
plt.title('Binary Classification Dataset')
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ── 함수 정의 ─────────────────────────────────────────────
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def bce_loss(y_pred, y_true):
    """Binary Cross-Entropy Loss."""
    eps = 1e-8   # log(0) 방지
    return -np.mean(y_true * np.log(y_pred + eps) +
                    (1 - y_true) * np.log(1 - y_pred + eps))

def predict(X, w, b):
    """Return probability P(y=1|X)."""
    return sigmoid(X @ w + b)

def accuracy(X, y, w, b, threshold=0.5):
    """Classification accuracy."""
    y_pred = predict(X, w, b)
    y_class = (y_pred >= threshold).astype(int)
    return np.mean(y_class == y)

# ── 파라미터 초기화 ───────────────────────────────────────
np.random.seed(0)
w = np.zeros((2, 1))   # (2, 1) — 특성 2개
b = 0.0
lr = 0.1
epochs = 300
loss_history = []
acc_history  = []

# ── 학습 루프 ─────────────────────────────────────────────
for epoch in range(epochs):

    # 1) Forward pass
    y_pred = predict(X_norm, w, b)          # sigmoid(Xw + b)

    # 2) Loss
    loss = bce_loss(y_pred, y)
    loss_history.append(loss)
    acc_history.append(accuracy(X_norm, y, w, b))

    # 3) Gradient
    diff = y_pred - y                        # (100, 1)
    dw   = (1 / N) * X_norm.T @ diff        # (2, 1)
    db   = np.mean(diff)

    # 4) Update
    w = w - lr * dw
    b = b - lr * db

    if (epoch + 1) % 50 == 0:
        print(f'Epoch {epoch+1:3d} | Loss: {loss:.4f} | Accuracy: {acc_history[-1]*100:.1f}%')

print(f'\nFinal Accuracy: {accuracy(X_norm, y, w, b)*100:.1f}%')

In [ ]:
# 학습 곡선
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(loss_history, color='steelblue', linewidth=2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('BCE Loss')
axes[0].set_title('Training Curve — Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot([a*100 for a in acc_history], color='seagreen', linewidth=2)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training Curve — Accuracy')
axes[1].set_ylim(40, 105)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 5. 결정경계 시각화 (Decision Boundary)

### 결정경계란?

$\hat{y} = \sigma(w_1 x_1 + w_2 x_2 + b) = 0.5$ 가 되는 선입니다.  
$\sigma(z) = 0.5$ 는 $z = 0$ 일 때이므로:

$$w_1 x_1 + w_2 x_2 + b = 0$$

이 식을 $x_2$ 에 대해 풀면:

$$x_2 = -\frac{w_1}{w_2} x_1 - \frac{b}{w_2}$$

로지스틱 회귀의 결정경계는 항상 **직선(선형)** 입니다.  
→ 나중에 MLP에서는 이 경계가 **곡선**이 됩니다!

In [ ]:
def plot_decision_boundary(X, y, w, b, X_mean, X_std, title='Decision Boundary'):
    """Plot decision boundary for logistic regression."""
    # 격자 생성
    x1_min, x1_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    x2_min, x2_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx1, xx2 = np.meshgrid(np.linspace(x1_min, x1_max, 200),
                            np.linspace(x2_min, x2_max, 200))

    # 격자 예측 (정규화 적용)
    grid = np.c_[xx1.ravel(), xx2.ravel()]
    grid_norm = (grid - X_mean) / X_std
    probs = sigmoid(grid_norm @ w + b).reshape(xx1.shape)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Left: decision boundary
    axes[0].contourf(xx1, xx2, probs, levels=50,
                     cmap='RdBu_r', alpha=0.6, vmin=0, vmax=1)
    axes[0].contour(xx1, xx2, probs, levels=[0.5],
                    colors='black', linewidths=2)
    axes[0].scatter(X[y.flatten()==0, 0], X[y.flatten()==0, 1],
                    color='tomato', label='Fail (y=0)', edgecolors='k', s=50)
    axes[0].scatter(X[y.flatten()==1, 0], X[y.flatten()==1, 1],
                    color='steelblue', label='Pass (y=1)', edgecolors='k', s=50)
    axes[0].set_xlabel('Math score'); axes[0].set_ylabel('English score')
    axes[0].set_title(title + '\n(black line = decision boundary)')
    axes[0].legend()

    # Right: probability map
    im = axes[1].contourf(xx1, xx2, probs, levels=50,
                          cmap='RdBu_r', alpha=0.9, vmin=0, vmax=1)
    plt.colorbar(im, ax=axes[1], label='P(y=1|x)')
    axes[1].contour(xx1, xx2, probs, levels=[0.5],
                    colors='black', linewidths=2)
    axes[1].set_xlabel('Math score'); axes[1].set_ylabel('English score')
    axes[1].set_title('Predicted Probability Map')

    plt.tight_layout()
    plt.show()

plot_decision_boundary(X, y, w, b, X_mean, X_std,
                       title='Logistic Regression Decision Boundary')

In [ ]:
# 로지스틱 회귀의 한계 — XOR 문제 (선형으로 분리 불가)
np.random.seed(1)
X_xor = np.random.randn(200, 2)
y_xor = ((X_xor[:, 0] > 0) ^ (X_xor[:, 1] > 0)).astype(int).reshape(-1, 1)

# 동일하게 학습
X_xor_norm = (X_xor - X_xor.mean(0)) / X_xor.std(0)
w_xor, b_xor = np.zeros((2, 1)), 0.0
for _ in range(500):
    yp = sigmoid(X_xor_norm @ w_xor + b_xor)
    dw = (1/200) * X_xor_norm.T @ (yp - y_xor)
    db = np.mean(yp - y_xor)
    w_xor -= 0.1 * dw; b_xor -= 0.1 * db

acc_xor = accuracy(X_xor_norm, y_xor, w_xor, b_xor)

plot_decision_boundary(X_xor, y_xor, w_xor, b_xor,
                       X_xor.mean(0), X_xor.std(0),
                       title=f'XOR Problem — Logistic Regression fails!\n(Accuracy: {acc_xor*100:.1f}%)')

print(f'XOR accuracy: {acc_xor*100:.1f}%  ← 50% = random!')
print('→ 직선으로는 XOR을 풀 수 없습니다.')
print('→ 이것이 MLP(다층 신경망)가 필요한 이유입니다!')

---
## 6. Softmax — 다중 클래스로 확장

### 이진분류 → 다중 클래스

이진분류는 출력이 하나(클래스 1일 확률)였습니다.  
클래스가 3개 이상이면? → **각 클래스일 확률을 동시에 출력**해야 합니다.

예: 손글씨 숫자 인식 (0~9, 10개 클래스)
- 출력: `[0.01, 0.02, 0.85, 0.03, ...]` — 가장 높은 것이 예측 클래스
- 조건: 모든 확률의 합 = 1

### Softmax 정의

$$\text{softmax}(z_k) = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}$$

- 모든 출력을 양수로 만들고 (지수함수)
- 합이 1이 되도록 정규화

### Sigmoid와의 관계

클래스가 2개일 때 Softmax를 적용하면 Sigmoid와 **수학적으로 동일**합니다.  
Softmax는 Sigmoid의 일반화 버전입니다.

In [ ]:
def softmax(z):
    """Numerically stable softmax."""
    z_shifted = z - z.max(axis=1, keepdims=True)   # overflow 방지
    exp_z = np.exp(z_shifted)
    return exp_z / exp_z.sum(axis=1, keepdims=True)

# 예시: 3개 클래스 (고양이 / 개 / 새)
# 모델이 출력한 원시 점수 (logit)
logits = np.array([[3.0, 1.0, 0.5],    # sample 1
                   [0.5, 2.5, 1.0],    # sample 2
                   [1.0, 1.0, 4.0]])   # sample 3

probs = softmax(logits)

print('Logits (raw scores):')
print(logits)
print('\nSoftmax probabilities:')
for i, (row, cls) in enumerate(zip(probs, ['Cat/Dog/Bird']*3)):
    pred = ['Cat', 'Dog', 'Bird'][np.argmax(row)]
    print(f'  Sample {i+1}: cat={row[0]:.3f}  dog={row[1]:.3f}  bird={row[2]:.3f}  → Predicted: {pred}')
print('\nRow sums (must be 1.0):', probs.sum(axis=1))

# 시각화
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
labels = ['Cat', 'Dog', 'Bird']
colors = ['steelblue', 'seagreen', 'tomato']
for i, ax in enumerate(axes):
    bars = ax.bar(labels, probs[i], color=colors, edgecolor='k', alpha=0.8)
    ax.set_ylim(0, 1)
    ax.set_ylabel('Probability')
    ax.set_title(f'Sample {i+1}')
    ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
    for bar, val in zip(bars, probs[i]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')
plt.suptitle('Softmax Output — Multi-class Probabilities', fontweight='bold')
plt.tight_layout()
plt.show()

### 이번 주 핵심 정리

| | 선형회귀 (1주차) | 로지스틱 회귀 (이번 주) | 다중 클래스 (이후) |
|---|---|---|---|
| **출력 함수** | 없음 ($\hat{y} = Xw+b$) | Sigmoid | Softmax |
| **Loss** | MSE | BCE | Cross-Entropy |
| **출력값** | 임의의 실수 | 0~1 (확률) | K개의 확률 (합=1) |
| **용도** | 숫자 예측 | 이진분류 | 다중 클래스 분류 |

**학습 루프 구조는 모두 동일합니다.** 달라지는 것은 출력 함수와 Loss뿐입니다.

### 다음 주 예고
> 지금까지는 **한 층짜리** 모델이었습니다.  
> 다음 주부터는 여러 층을 쌓는 **Neural Network(신경망)** 을 시작합니다.  
> 먼저 **PyTorch**를 소개하고, 지금까지 NumPy로 짠 코드를 PyTorch로 옮겨봅니다.


---
---

# Part 2. 실습 문제

> **[A] 한 줄 채우기** — 8문제  
> **[B] 여러 줄 채우기** — 2문제 (힌트 제공)
>
> - `______` 로 표시된 부분만 채우세요
> - 막히면 위 Part 1의 같은 개념 코드를 참고하세요
> - [B] 문제 중 못 한 것은 숙제입니다

---
## [A] 한 줄 채우기

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────
# A-1. 시그모이드 함수 구현
# σ(z) = 1 / (1 + e^{-z})
# Expected: sigmoid(0) = 0.5
# ─────────────────────────────────────────────
def sigmoid(z):
    return 1 /(1 + np.exp(-z)) 

print('sigmoid(-2):', sigmoid(-2))   # ≈ 0.119
print('sigmoid( 0):', sigmoid(0))    # = 0.5
print('sigmoid( 2):', sigmoid(2))    # ≈ 0.881

In [ ]:
# ─────────────────────────────────────────────
# A-2. 시그모이드 미분
# σ'(z) = σ(z) * (1 - σ(z))
# Expected: sigmoid_grad(0) = 0.25
# ─────────────────────────────────────────────
def sigmoid_grad(z):
    s = sigmoid(z)
    return s * (1 - s)

print('sigmoid_grad(0):', sigmoid_grad(0))    # = 0.25
print('sigmoid_grad(5):', sigmoid_grad(5))    # ≈ 0.007 (vanishing!)

In [ ]:
# ─────────────────────────────────────────────
# A-3. Forward pass
# y_pred = sigmoid(X @ w + b)
# Expected shape: (5, 1), values in (0, 1)
# ─────────────────────────────────────────────
np.random.seed(0)
X_test = np.random.randn(5, 3)
w_test = np.random.randn(3, 1)
b_test = 0.0

y_pred_test = sigmoid(X_test @ w_test + b_test)
print('y_pred shape:', y_pred_test.shape)   # (5, 1)
print('y_pred:\n', y_pred_test)

In [ ]:
# ─────────────────────────────────────────────
# A-4. BCE Loss 계산
# L = -mean(y*log(y_pred) + (1-y)*log(1-y_pred))
# Expected: ≈ 0.693 (random prediction → log(0.5))
# ─────────────────────────────────────────────
y_true_test = np.array([[1], [0], [1], [1], [0]])
y_pred_bce  = np.array([[0.5], [0.5], [0.5], [0.5], [0.5]])  # 아무것도 모를 때
eps = 1e-8  # log(0)을 방지하는 코드

loss_test = -np.mean(y_true_test * np.log(y_pred_bce + eps) + (1 - y_true_test) * np.log(1 - y_pred_bce + eps)) # eps을 더해 무한으로 발산하는 log(0)을 방지
print('BCE Loss (random prediction):', loss_test)   # ≈ 0.693

In [ ]:
# ─────────────────────────────────────────────
# A-5. Gradient 계산
# dw = (1/N) * X.T @ (y_pred - y)
# Expected shape: (3, 1)
# ─────────────────────────────────────────────
N_test = 5
diff_test = y_pred_test - y_true_test

dw_test = (1 / N_test) * X_test.T @ diff_test
print('dw shape:', dw_test.shape)   # (3, 1)

In [ ]:
# ─────────────────────────────────────────────
# A-6. 예측 클래스 결정
# 확률이 0.5 이상이면 1, 미만이면 0
# Expected: [1, 0, 1, 0, 1]
# ─────────────────────────────────────────────
probs = np.array([0.8, 0.3, 0.6, 0.2, 0.9])

predictions = (probs >= 0.5).astype(int) # 그대로 출력할시 boolean이여서 true, false로 출력. astype(int)를 통해 1과 0으로 변환
print('Predictions:', predictions)

In [ ]:
# ─────────────────────────────────────────────
# A-7. Accuracy 계산
# Expected: 0.8  (5개 중 4개 맞음)
# ─────────────────────────────────────────────
y_true_acc = np.array([1, 0, 1, 0, 1])
y_pred_acc = np.array([1, 0, 1, 1, 1])   # 4번째만 틀림

acc = np.mean(y_pred_acc == y_true_acc)
print('Accuracy:', acc)   # 0.8

In [ ]:
# ─────────────────────────────────────────────
# A-8. Softmax 구현
# softmax(z_k) = exp(z_k) / sum(exp(z))
# Expected: values in (0,1), row sums = 1.0
# ─────────────────────────────────────────────
def softmax(z):
    z_shifted = z - z.max(axis=1, keepdims=True)   # numerical stability
    exp_z = np.exp(z_shifted)
    return exp_z / exp_z.sum(axis = 1, keepdims = True)

z_test = np.array([[2.0, 1.0, 0.5],
                   [1.0, 3.0, 0.2]])
probs_test = softmax(z_test)
print('Softmax output:\n', probs_test.round(4))
print('Row sums:', probs_test.sum(axis=1))   # [1.0, 1.0]

---
## [B] 여러 줄 채우기

In [ ]:
# ─────────────────────────────────────────────
# B-1. 로지스틱 회귀 학습 루프 완성
#
# Hint:
#   forward : y_pred = sigmoid(X_norm @ w + b)
#   loss    : bce_loss(y_pred, y)
#   gradient: dw = (1/N) * X_norm.T @ (y_pred - y)
#             db = np.mean(y_pred - y)
#   update  : w = w - lr * dw
# ─────────────────────────────────────────────
np.random.seed(42)
N = 80

# 데이터: 나이(age)와 종양 크기(size)로 악성(1) / 양성(0) 예측
age_m  = np.random.randn(N//2) * 8 + 55   # 악성: 나이 높음
age_b  = np.random.randn(N//2) * 8 + 40   # 양성: 나이 낮음
size_m = np.random.randn(N//2) * 0.5 + 3  # 악성: 크기 큼
size_b = np.random.randn(N//2) * 0.5 + 1.5  # 양성: 크기 작음

X_b1 = np.column_stack([np.r_[age_m, age_b], np.r_[size_m, size_b]])
y_b1 = np.array([1]*(N//2) + [0]*(N//2)).reshape(-1, 1)

# 정규화
X_b1_norm = (X_b1 - X_b1.mean(0)) / X_b1.std(0)

# helper functions
def sigmoid(z): return 1 / (1 + np.exp(-z))
def bce_loss(yp, yt):
    eps = 1e-8
    return -np.mean(yt * np.log(yp + eps) + (1-yt) * np.log(1-yp+eps))

# 파라미터 초기화
w_b1, b_b1 = np.zeros((2, 1)), 0.0
lr_b1 = 0.1
loss_hist, acc_hist = [], []

for epoch in range(300):

    # Forward
    y_pred_b1 = sigmoid(X_b1_norm @ w_b1 + b_b1)  # (80, 1)

    # Loss
    loss_b1 = bce_loss(y_pred_b1, y_b1)
    loss_hist.append(loss_b1)

    # Accuracy
    preds = (y_pred_b1 >= 0.5).astype(int)
    acc_hist.append(np.mean(preds == y_b1))

    # Gradient
    diff = y_pred_b1 - y_b1
    dw_b1 = (1/N) * X_b1_norm.T @ diff
    db_b1 = np.mean(diff)

    # Update
    w_b1 = w_b1 - lr_b1 * dw_b1
    b_b1 = b_b1 - lr_b1 * db_b1

print(f'Final Loss: {loss_hist[-1]:.4f}')
print(f'Final Accuracy: {acc_hist[-1]*100:.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(loss_hist, color='steelblue')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('BCE Loss')
axes[0].set_title('B-1 Training Loss'); axes[0].grid(True, alpha=0.3)
axes[1].plot([a*100 for a in acc_hist], color='seagreen')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('B-1 Training Accuracy'); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ─────────────────────────────────────────────
# B-2. 결정경계 그리기
#
# B-1에서 학습한 w_b1, b_b1을 이용해 결정경계를 시각화하세요
#
# Hint:
#   격자 생성:  xx1, xx2 = np.meshgrid(x1_range, x2_range)
#   격자 예측:  grid_norm = (grid - X_b1.mean(0)) / X_b1.std(0)
#               probs = sigmoid(grid_norm @ w_b1 + b_b1).reshape(xx1.shape)
#   경계선:     plt.contour(..., levels=[0.5])
# ─────────────────────────────────────────────

# 격자 범위 설정
x1_min, x1_max = X_b1[:, 0].min()-2, X_b1[:, 0].max()+2  # age 범위
x2_min, x2_max = X_b1[:, 1].min()-1, X_b1[:, 1].max()+1  # size 범위

xx1, xx2 = np.meshgrid(np.linspace(x1_min, x1_max, 200),
                        np.linspace(x2_min, x2_max, 200))

# 격자 예측
grid = np.c_[xx1.ravel(), xx2.ravel()]
grid_norm = (grid - X_b1.mean(0)) / X_b1.std(0)
probs_b2  = sigmoid(grid_norm @ w_b1 + b_b1)
probs_b2  = probs_b2.reshape(xx1.shape)

# 시각화
plt.figure(figsize=(7, 5))
plt.contourf(xx1, xx2, probs_b2, levels=50, cmap='RdBu_r', alpha=0.7, vmin=0, vmax=1)

# TODO: 결정경계(확률=0.5 등고선)를 검은 실선으로 그리세요
plt.contour(xx1, xx2, probs_b2, levels=[0.5], colors='black', linewidths=2)

plt.scatter(X_b1[y_b1.flatten()==0, 0], X_b1[y_b1.flatten()==0, 1],
            color='tomato', label='Benign (y=0)', edgecolors='k', s=50)
plt.scatter(X_b1[y_b1.flatten()==1, 0], X_b1[y_b1.flatten()==1, 1],
            color='steelblue', label='Malignant (y=1)', edgecolors='k', s=50)
plt.xlabel('Age'); plt.ylabel('Tumor Size')
plt.title('B-2 Decision Boundary')
plt.legend(); plt.grid(True, alpha=0.2)
plt.show()

---
## Checklist

**[A] 한 줄 채우기**
- [ ] A-1. sigmoid 구현
- [ ] A-2. sigmoid 미분
- [ ] A-3. forward pass
- [ ] A-4. BCE loss 계산
- [ ] A-5. gradient 계산
- [ ] A-6. 예측 클래스 결정
- [ ] A-7. accuracy 계산
- [ ] A-8. softmax 구현

**[B] 여러 줄 채우기**
- [ ] B-1. 로지스틱 회귀 학습 루프 완성
- [ ] B-2. 결정경계 시각화 ← 못 한 경우 숙제

---
## 생각해볼 질문

1. A-2에서 `sigmoid_grad(5)` 가 매우 작은 값이 나왔습니다. 이것이 나중에 깊은 신경망에서 어떤 문제를 일으킬까요? (**Vanishing Gradient** 키워드)
2. B-2의 결정경계는 직선입니다. 데이터가 원형으로 분포해 있다면 로지스틱 회귀로 잘 분류할 수 있을까요?
3. Softmax 출력의 합이 항상 1이 되는 이유를 수식으로 설명해보세요.